# Kafka Streaming

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LineageLogic/LakeLogic/blob/main/examples/03_data_sources/streaming/kafka/kafka_demo.ipynb) 
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/LineageLogic/LakeLogic/blob/main/examples/03_data_sources/streaming/kafka/kafka_demo.ipynb)

## Business Scenario

Kafka topics power real-time data pipelines. You need to validate messages at ingestion and avoid corrupting downstream tables.

## Value Proposition

- Contract-driven validation for Kafka events
- Quarantine invalid messages
- Reliable ingestion into lakehouse storage

---

## Goals

1. Connect to a Kafka topic
2. Validate and process messages
3. Store clean events


## 🚀 Step 1: Setup Kafka

Before running this notebook, you need:
1. A running Kafka cluster (local, Confluent Cloud, AWS MSK, etc.)
2. A Kafka topic (e.g., `user-events`)
3. Broker addresses

For local testing, you can use Docker:
```bash
docker run -d --name kafka -p 9092:9092 apache/kafka:latest
```

## 📝 Step 2: Review the Contract

Our contract defines the expected Kafka message schema and quality rules.

In [ ]:
with open('kafka_contract.yaml', 'r') as f:
    print("📄 Kafka Contract:")
    print("-----------------")
    print(f.read())

## ▶️ Step 3: Start the Kafka Consumer

This will connect to your Kafka topic and start processing messages.

In [ ]:
from lakelogic.core.streaming_processor import StreamingDataProcessor
import threading
import time

# Initialize processor
processor = StreamingDataProcessor(
    contract="kafka_contract.yaml",
    framework="bytewax"
)

# Start in background
thread = threading.Thread(target=processor.start)
thread.daemon = True
thread.start()

print("🚀 Kafka consumer started!")
print("Waiting for user events...")
time.sleep(5)

## 🧪 Step 4: Produce Test Messages

Let's produce some test user events to the Kafka topic.

In [ ]:
from kafka import KafkaProducer
import json
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

test_events = [
    {
        "userId": "USER-123456",
        "eventType": "login",
        "eventData": {"device": "mobile", "os": "iOS"},
        "timestamp": datetime.now().isoformat(),
        "sessionId": "SESSION-001",
        "ipAddress": "192.168.1.100"
    },
    {
        "userId": "USER-789012",
        "eventType": "purchase",
        "eventData": {"productId": "PROD-456", "amount": 99.99},
        "timestamp": datetime.now().isoformat(),
        "sessionId": "SESSION-002"
    },
    {
        "userId": "USER-345678",
        "eventType": "search",
        "eventData": {"query": "wireless headphones"},
        "timestamp": datetime.now().isoformat()
    }
]

for event in test_events:
    future = producer.send('user-events', value=event)
    result = future.get(timeout=10)
    print(f"📤 Produced {event['eventType']} event for {event['userId']} - Partition: {result.partition}, Offset: {result.offset}")

producer.flush()
print("\n✅ Test messages produced! Check LakeLogic logs...")

## 📊 Step 5: Verify Results

Check the materialized Delta table.

In [ ]:
import polars as pl
import time

# Wait for processing
time.sleep(3)

try:
    df = pl.read_delta("./data/bronze/kafka_events/")
    print("📂 Processed Events:")
    print(df)
    
    print("\n📊 Event Type Distribution:")
    print(df.group_by("eventType").count())
except Exception as e:
    print(f"ℹ️  No data yet: {e}")

## 🎉 Summary

You just:
- ✅ Connected to Apache Kafka
- ✅ Validated user events against a contract
- ✅ Materialized events to Delta Lake

This pattern enables **enterprise-grade real-time analytics** with quality guarantees!